In [ ]:
import matplotlib
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import cupy
import os
from sklearn.model_selection import cross_val_score

kaggle = True if os.environ.get('KAGGLE_URL_BASE','') else False
selected_features = True
inliers = True

if kaggle:
    training_data = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
else:
    training_data = 'data/train.csv'

df_tv = pd.read_csv(training_data)
df_x = df_tv.iloc[:,1:-1]

df_dummy = pd.get_dummies(df_x, dtype=int, drop_first=False)

if selected_features:
    df_label = df_tv.replace({'Irrigation_Need': {'Low': 0, 'Medium': 1, 'High': 2}})['Irrigation_Need']
    df_full = pd.concat([df_dummy, df_label], axis=1)
    corr = df_full.corr()
    df_corr_sort_abs = corr.abs().sort_values(by='Irrigation_Need', ascending=False)['Irrigation_Need']
    threshold = 0.1
    df_dummy = df_full[df_corr_sort_abs[df_corr_sort_abs > threshold].index]
    df_dummy.drop(columns=['Irrigation_Need'], inplace=True)

x = df_dummy.iloc[:,:].values
y = df_tv.iloc[:,-1].values

class_le = LabelEncoder()
y = class_le.fit_transform(y)

In [ ]:

from lightgbm import LGBMClassifier

# device='cuda' is only supported in lightgbm built with: 
# pip install lightgbm --install-option=cmake.define.USE_CUDA=ON
lgbm = LGBMClassifier(class_weight="balanced", n_estimators=500, 
                      learning_rate=0.1, max_depth=3, reg_lambda=1,
                      boosting_type='gbdt', is_unbalance=True,objective='multiclass', 
                      random_state=1, verbose=-1)
                      #device='cuda')
#lgbm.fit(x, y)
#cv_scores = cross_val_score(lgbm, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

In [ ]:
from catboost import CatBoostClassifier

params = {'iterations': 2000,
          'depth': 3,
          'learning_rate': .1,
          'loss_function': 'MultiClass',
          'verbose': False,
          'task_type': 'GPU',
          'reg_lambda': 0.1,
          'auto_class_weights': 'Balanced',
          'random_state': 1}
          #'class_weights': weights}
#cb = CatBoostClassifier(**params)
                        
from sklearn.base import clone

class CustomCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return CustomCatBoostClassifier(**self.get_params())
    
cb = CustomCatBoostClassifier(**params)
#cb.fit(x, y)
#cv_scores = cross_val_score(cb, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(class_weight='balanced', max_iter=500, 
                                     learning_rate=0.1, max_depth=3, l2_regularization=0.1,
                                     loss='log_loss', random_state=1)

#cv_scores = cross_val_score(hgb, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[('hgb', hgb), ('lgbm', lgbm), ('cb', cb)], voting='soft')
#cv_scores = cross_val_score(voting_clf, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")
voting_clf.fit(x, y)

In [ ]:
if kaggle:
    testing_data = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
else:
    testing_data = 'data/test.csv'

df_test = pd.read_csv(testing_data)

ids = df_test['id'].values

df_test_dummy = pd.get_dummies(df_test.iloc[:,1:], drop_first=False, dtype=int)
df_test_dummy = df_test_dummy[df_dummy.columns]
x_test = df_test_dummy.to_numpy()

In [ ]:

if kaggle:
    out_dir = '/kaggle/working/'
else:
    out_dir = 'data/'

df_submission_stack = pd.DataFrame({'id': ids, 'Irrigation_Need': class_le.inverse_transform(voting_clf.predict(x_test))})
df_submission_stack.to_csv(os.path.join(out_dir, 'submission-voting_clf_tuned_select_features_inliers_v1.csv'), index=False)